# CAPTOR on natural images

A frozen DINOv2 ViT-B/14 (with registers) read at 196 pixels, and a table counted
from the captions of 15,000 COCO training images. No text encoder is used at any
point: each term is a row of the table, and the score of a term at a patch is
read from the visual words the patch falls into.

The table (about 67 MB) and the DINOv2 weights are downloaded on first use.
Everything runs on the CPU.

In [ ]:
# Setup for Google Colab: fetch the code and the example images (skipped when run locally)
import os, sys
if "google.colab" in sys.modules:
    if not os.path.isdir("/content/captor-repo"):
        !git clone -q https://github.com/EladYoshai/captor.git /content/captor-repo
        !pip install -q -e /content/captor-repo
    %cd /content/captor-repo/examples
    sys.path.insert(0, "/content/captor-repo")


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image

from captor import CAPTOR
from captor.viz import crop_like_encoder, heatmap, smooth_overlay

model = CAPTOR.from_pretrained("coco_dinov2reg_k640")
print(len(model.terms), "terms:", ", ".join(model.terms[:12]), "...")

## Name every patch with one of a few terms

List the COCO objects you expect in the photo. Every patch is named with one of
them, so the sky or the ground still receives the closest object.
`smooth_overlay` upsamples the patch scores to pixels before picking each pixel's
term, and with `fade=0.3` it leaves pixels where no listed term scores well
uncoloured.

In [ ]:
from matplotlib.patches import Patch
from captor.viz import legend

image = Image.open(Path("images") / "giraffe_zebra.jpg")
terms = ["giraffe", "zebra"]
seg = model.segment(image, terms)
keep = seg.confident(0.3)

view = crop_like_encoder(image, center_crop=True)   # DINOv2 reads the centre square
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(view); ax[1].imshow(smooth_overlay(view, seg.scores, fade=0.3))
ax[1].legend(handles=[Patch(color=[v / 255 for v in col], label=t) for t, col in legend(terms)],
             loc="lower right")
for a in ax: a.axis("off")

## One row of the score matrix

`seg.scores[..., i]` is the score of term `i` at every patch, the quantity a
cross-attention layer would pool over.

In [ ]:
fig, ax = plt.subplots(1, len(terms), figsize=(4 * len(terms), 4.2))
for i, t in enumerate(terms):
    ax[i].imshow(Image.blend(view.convert("RGB"), heatmap(seg.scores[..., i], view.size), 0.6))
    ax[i].set_title(t); ax[i].axis("off")

## More examples

The middle row names the patches and the bottom row is the soft score map of the
first listed term.

In [ ]:
examples = {"beagle_sofa.jpg": ["dog"],
            "elephants.jpg": ["elephant"],
            "cyclist.jpg": ["bicycle", "person", "car"],
            "street.jpg": ["bicycle", "person", "car"],
            "breakfast.jpg": ["dining table", "sandwich", "cup", "bottle"]}
fig, ax = plt.subplots(3, len(examples), figsize=(3.6 * len(examples), 12))
for c, (name, terms) in enumerate(examples.items()):
    image = Image.open(Path("images") / name)
    seg = model.segment(image, terms)
    view = crop_like_encoder(image, center_crop=True)
    ax[0, c].imshow(view); ax[0, c].set_title(name)
    ax[1, c].imshow(smooth_overlay(view, seg.scores, fade=0.3))
    ax[1, c].legend(handles=[Patch(color=[v / 255 for v in col], label=t)
                             for t, col in legend(terms)],
                    loc="upper center", bbox_to_anchor=(0.5, -0.01), ncol=2, frameon=False)
    # soft score map of the first term, as in the cell above
    ax[2, c].imshow(Image.blend(view.convert("RGB"), heatmap(seg.scores[..., 0], view.size), 0.6))
    ax[2, c].set_title(f"score map: {terms[0]}")
for a in ax.ravel(): a.axis("off")
fig.subplots_adjust(hspace=0.3)

## Ranking whole images (Eq. score)

In [ ]:
names = ["giraffe_zebra.jpg", "beagle_sofa.jpg", "elephants.jpg", "cyclist.jpg", "breakfast.jpg"]
query = ["giraffe", "dog", "elephant", "bicycle", "cup"]
for name in names:
    s = model.score(Path("images") / name, query)
    print(f"{name:<18}", "  ".join(f"{t} {v:+6.1f}" for t, v in s.items()))